# 延后初始化与参数推断

本笔记改写自原 `deferred-init.ipynb`,介绍在 PyTorch 中根据真实数据形状延后(懒惰)初始化模型参数的技巧,并结合实践说明其优势与注意事项。

## 1. 为什么需要延后初始化?

- 某些模型结构在构建时不易确定输入维度(如根据数据动态生成层数)。
- 在处理变长序列或动态计算图时,提前指定尺寸既繁琐又容易出错。
- 延后初始化允许根据第一批真实数据推断参数形状,减少硬编码。

## 2. 手动推断 vs 自动推断

| 方式 | 优点 | 缺点 |
|------|------|------|
| 手动指定 | 明确可控 | 需要计算形状,易出错 |
| 延后初始化 | 代码更简洁,更易复用 | 需在第一次前向后才能访问参数 |

## 3. PyTorch 中的延后组件

- `torch.nn.LazyLinear`
- `torch.nn.LazyConv2d`
- `torch.nn.LazyBatchNorm1d/2d`

这些层会在第一次前向传播时根据输入数据自动推断 `in_features`/`in_channels` 等参数。

## 4. 实战示例



In [ ]:
import torch
from torch import nn

torch.manual_seed(0)

# 使用 LazyLinear 构建全连接网络
net = nn.Sequential(
    nn.LazyLinear(128),
    nn.ReLU(),
    nn.LazyLinear(10)
)

print('初始化前的参数状态:')
for name, param in net.named_parameters():
    print(name, param.shape, param.numel())

x = torch.randn(16, 64)  # 输入的真实形状在运行时才确定
out = net(x)
print('
前向传播后的输出形状:', out.shape)

print('
初始化后的参数状态:')
for name, param in net.named_parameters():
    print(name, param.shape, param.numel())



## 5. 延后初始化内部机制

- 第一次调用 `forward` 时,层会读取输入张量的最后一维(线性层)或通道数(卷积层)。
- 根据推断得到的维度创建权重与偏置张量,并调用默认初始化方法。
- 后续前向传播复用已创建的参数。

## 6. 自定义模块中的延后初始化

如果自定义模块需要延后初始化,可以重写 `reset_parameters` 并在前向传播时检查参数是否存在。



In [ ]:
class LazyMLP(nn.Module):
    def __init__(self, hidden_dim=128, num_classes=10):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.fc1 = None
        self.fc2 = None

    def forward(self, x):
        if self.fc1 is None:
            in_dim = x.shape[-1]
            self.fc1 = nn.Linear(in_dim, self.hidden_dim).to(x.device)
            self.fc2 = nn.Linear(self.hidden_dim, self.num_classes).to(x.device)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

lazy_net = LazyMLP()
print('初始化前参数列表:', list(lazy_net.parameters()))

x = torch.randn(8, 20)
out = lazy_net(x)
print('输出形状:', out.shape)
print('初始化后参数数量:', sum(p.numel() for p in lazy_net.parameters()))



## 7. 注意事项

1. **延后层不可序列化为空权重**: 在保存模型前必须至少运行一次前向传播,否则 `state_dict()` 中缺少参数。
2. **与 `torch.jit`/ONNX**: 部分延后层在脚本化或导出时需要手动指定 `example_input`。
3. **分布式训练**: 多卡训练需确保所有进程在初始化前同步,通常通过广播第一批数据实现。
4. **梯度检查**: 延后初始化不会影响自动求导,但调试时需确认已执行前向传播。

## 8. 练习

1. 将 `LazyConv2d` 用于 `06_卷积神经网络/02_LeNet实战` 中的模型,观察参数推断效果。
2. 尝试在延后初始化后使用自定义初始化函数(`nn.init.xavier_uniform_`)。
3. 探索如何在保存模型时确保延后层的参数被写入 `state_dict`。

---
延后初始化可以显著提高模型代码的灵活性,建议在需要动态输入尺寸或快速原型时使用。下一步可继续学习 `02_GPU加速计算.ipynb`,了解如何在推断后的网络上部署到 GPU。

